# Lab 7 — Large Language Models (LLMs)



## The Three Main Transformer Model Families

| Family | Example Models | Main Use |
|---|---|---|
| **Encoder-only** | BERT, RoBERTa | Classification, sentiment analysis, NER |
| **Encoder–decoder** | T5, BART | Translation, summarization, question answering |
| **Decoder-only** | GPT family | Text generation, conversation, completion |


## 1. Setup


In [2]:
!pip -q install transformers sentencepiece torch


In [3]:
from transformers import pipeline, set_seed

print("Libraries loaded successfully!")


Libraries loaded successfully!


# Part A — Decoder-Only Model

## Decoder-only example: GPT-style Text Generation

A decoder-only model predicts the next token based on previous tokens.

We will use **DistilGPT-2**, a small GPT-style model.


In [19]:
generator = pipeline("text-generation", model="distilgpt2")

prompt = "Natural language processing is"

output = generator(prompt, max_new_tokens=30, do_sample=False)

print(output[0]["generated_text"])

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Natural language processing is a common feature of the language processing process.























# Part B — Encoder–Decoder Model

## Encoder–decoder example: Instruction / Text-to-Text Generation

Encoder–decoder models contain two main parts:

1. The **encoder** reads the input.
2. The **decoder** produces the output.

We will use **FLAN-T5-small**.


In [24]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

prompt = "Translate to French: I love natural language processing."

# Convert text into tokens
output = model.generate(**tokenizer(prompt, return_tensors="pt"))

# Convert tokens back to text
print(tokenizer.decode(output[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Je s'agissant de la technologie de linguage naturelle.


# Part C — Encoder-Only Model

## Encoder-only example: Sentiment Classification

Encoder-only models read the full input and create a representation of it.

They are very useful when the task is to **understand or classify text**.

Here we use a DistilBERT sentiment model.


In [20]:
classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

text = "This NLP lab is easy and interesting."

result = classifier(text)
print(result)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.999778687953949}]


# Tasks

Complete the following tasks.

## Task 1 — Model Families

For each task below, choose the most suitable model family:

1. Sentiment classification
2. Machine translation
3. Open-ended text generation
4. Named Entity Recognition
5. Text summarization

## Task 1 — Answers

| # | Task | Most suitable family | Example model |
|---|---|---|---|
| 1 | Sentiment classification | **Encoder-only** | BERT, DistilBERT |
| 2 | Machine translation | **Encoder–decoder** | T5, BART |
| 3 | Open-ended text generation | **Decoder-only** | GPT family |
| 4 | Named Entity Recognition | **Encoder-only** | BERT, RoBERTa |
| 5 | Text summarization | **Encoder–decoder** | T5, BART |

**Reasoning:** Classification and NER need a full understanding of the input (encoder-only reads the whole sentence in both directions). Translation and summarization transform one text into another (encoder reads the input, decoder writes the output). Open-ended generation predicts the next token from previous tokens (decoder-only).

## Task 2 — Encoder-only Model

Use the sentiment classifier on these three sentences:

```text
The movie was excellent.
The lecture was difficult to understand.
I enjoyed learning about transformers.
```

Print the predicted label and score for each sentence.

In [4]:
!pip -q install transformers sentencepiece torch

In [5]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

sentences = [
    "The movie was excellent.",
    "The lecture was difficult to understand.",
    "I enjoyed learning about transformers."
]

results = classifier(sentences)

for sentence, result in zip(sentences, results):
    print(f"Sentence: {sentence}")
    print(f"  Label: {result['label']}")
    print(f"  Score: {result['score']:.4f}\n")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Sentence: The movie was excellent.
  Label: POSITIVE
  Score: 0.9999

Sentence: The lecture was difficult to understand.
  Label: NEGATIVE
  Score: 0.9997

Sentence: I enjoyed learning about transformers.
  Label: POSITIVE
  Score: 0.9997



## Task 3 — Encoder–Decoder Model

Use the FLAN-T5 model to perform the following:

- Summarize a short paragraph.


In [6]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

paragraph = (
    "Transformers are a type of neural network architecture introduced in 2017. "
    "They rely on a mechanism called self-attention, which allows the model to weigh "
    "the importance of different words in a sentence. Transformers have become the "
    "foundation of modern natural language processing, powering models such as BERT, "
    "T5, and GPT. They are used for tasks like translation, summarization, and text generation."
)

prompt = "Summarize: " + paragraph

inputs = tokenizer(prompt, return_tensors="pt")
output = model.generate(**inputs, max_new_tokens=50, num_beams=4, early_stopping=True)

print("Original paragraph:\n", paragraph)
print("\nSummary:\n", tokenizer.decode(output[0], skip_special_tokens=True))

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Original paragraph:
 Transformers are a type of neural network architecture introduced in 2017. They rely on a mechanism called self-attention, which allows the model to weigh the importance of different words in a sentence. Transformers have become the foundation of modern natural language processing, powering models such as BERT, T5, and GPT. They are used for tasks like translation, summarization, and text generation.

Summary:
 Transformers are a type of neural network that uses a mechanism called self-attention.


## Task 5 — Short Reflection

**Why is a decoder-only model suitable for a chatbot such as ChatGPT?**

A decoder-only model is suitable for a chatbot because it is trained to predict the next token from all the previous tokens, and a conversation needs exactly that: a reply generated word by word that continues the dialogue. The whole conversation history, meaning the user's messages and the earlier replies, can be fed in as one sequence. The model then uses causal (left-to-right) attention to write a coherent continuation. This makes it very flexible, because many tasks, such as answering questions, explaining, translating, or writing code, can all be framed as "continue this text." When it is trained on huge amounts of text and then fine-tuned with instructions and human feedback, a decoder-only model becomes very good at producing fluent, open-ended, human-like responses.

# Optional Challenge

Try generating text several times with the same prompt.

Remove:

```python
set_seed(42)
```

and run the sampling cell multiple times.

### Question

Why do you sometimes get different outputs even when the prompt is the same?

**Hint:** Sampling introduces randomness when choosing the next token.


In [7]:
from transformers import pipeline, set_seed

generator = pipeline("text-generation", model="distilgpt2")

set_seed(42)

prompt = "Natural language processing is"

for i in range(3):
    out = generator(prompt, max_new_tokens=30, do_sample=True, temperature=0.9, top_k=50,
                    pad_token_id=generator.tokenizer.eos_token_id)
    print(f"Run {i+1}: {out[0]['generated_text']}\n")

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  353MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'pad_token_id', 'do_sample', 'max_new_tokens', 'top_k'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[tran

Run 1: Natural language processing is more prevalent worldwide than is used in traditional languages, because it is harder to understand and understand these languages. The research team found that most languages are languages



[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Run 2: Natural language processing is an area of interest for both of us and the other species.

Run 3: Natural language processing is a fundamental part of the science and culture of the United States. As a result, our nation› has witnessed a profound increase in education, economic



In [8]:
from transformers import pipeline

generator = pipeline("text-generation", model="distilgpt2")

prompt = "Natural language processing is"

for i in range(3):
    out = generator(prompt, max_new_tokens=30, do_sample=True, temperature=0.9, top_k=50,
                    pad_token_id=generator.tokenizer.eos_token_id)
    print(f"Run {i+1}: {out[0]['generated_text']}\n")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Run 1: Natural language processing is a means of processing a language used by people in countries with no social networks. For the majority of people speaking in one language, language processing can be



[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Run 2: Natural language processing is done using JavaScript that is then applied in the standard library of JavaScript, as well as JavaScript runtime. In the event of a particular compiler change due to

Run 3: Natural language processing is often a matter of convenience to people of various national backgrounds. A very complex linguistic system, which can and should be used to describe language processing and its

